# Palm Oil Production Prediction
**Final Year Project (FYP1) Methodology - ML Implementation**

This notebook follows the methodology described in the provided FYP1 Methodology document to predict palm oil production using machine learning, based only on:
- Provided Excel datasets
- Python libraries: `scikit-learn`, `pandas`, `matplotlib`, `seaborn`

✅ PyCaret is not used. Due to execution issues, so it was discarded.


In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import ExtraTreesRegressor, AdaBoostRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

import warnings
warnings.filterwarnings('ignore')


In [2]:

# Load the two provided datasets correctly
area_df = pd.read_excel('Oil-Palm-Matured-Area.xlsx', sheet_name='Worksheet', skiprows=1)
yield_df = pd.read_excel('Palm-Oil-Yield.xlsx', sheet_name='Worksheet', skiprows=1)

# Preview both datasets
print("Matured Area Data:")
display(area_df.head())

print("\nPalm Oil Yield Data:")
display(yield_df.head())



Matured Area Data:


,Period,Indonesia,Malaysia,Thailand,Nigeria,Colombia,Ghana,Honduras,Papua New Guinea,Other Countries
0,2010,6262,4202,590,430,256,360,100,135,601
1,2011,6551,4282,620,450,273,382,105,140,859
2,2012,6990,4353,645,420,302,397,110,143,1537
3,2013,7856,4526,690,425,336,409,117,146,1280
4,2014,8130,4689,720,440,350,349,125,150,1597



Palm Oil Yield Data:


,Period,Indonesia,Malaysia,Nigeria,Thailand,Colombia,Ghana,Honduras,Papua New Guinea,Other Countries
0,2010,5456,2185,1639,1242,662,285,123,10,34922
1,2011,6349,2203,1733,1374,768,295,128,12,35904
2,2012,7128,2267,1860,1561,796,330,138,14,38486
3,2013,8033,2353,2161,1746,867,340,143,31,42264
4,2014,8643,2818,2294,1830,863,330,149,42,38561


In [6]:

# Merge datasets on 'Period' column
df = pd.merge(yield_df, area_df, on='Period', how='inner')

# Convert Period to datetime using January 1st of each year
df['Date'] = pd.to_datetime(df['Period'].astype(str) + '-01-01')

df = df.sort_values('Date').reset_index(drop=True)

# Sort chronologically
df.sort_values('Date', inplace=True)

# Drop rows with any missing values (or use imputation if needed)
df.dropna(inplace=True)

# Reset index
df.reset_index(drop=True, inplace=True)


In [10]:
# Fix reading of yield_df to get proper column names
raw_yield_df = pd.read_excel('Palm-Oil-Yield.xlsx', sheet_name='Worksheet', header=None)

# Use the second row (index 1) as column names
new_header = raw_yield_df.iloc[1]
yield_df = raw_yield_df[2:].copy()
yield_df.columns = new_header
yield_df.reset_index(drop=True, inplace=True)

# Ensure 'Period' column is numeric
yield_df['Period'] = pd.to_numeric(yield_df['Period'], errors='coerce')
yield_df.dropna(subset=['Period'], inplace=True)
yield_df['Period'] = yield_df['Period'].astype(int)

In [11]:
# Ensure dataframe is independent copy
df = df.copy()

# Add temporal features based on 'Period'
df['Date'] = pd.to_datetime(df['Period'].astype(str) + '-01-01')
df = df.sort_values('Date').reset_index(drop=True)
df['Year_sin'] = np.sin(2 * np.pi * df['Period'] / df['Period'].max())
df['Year_cos'] = np.cos(2 * np.pi * df['Period'] / df['Period'].max())

# Average Yield across all countries
yield_country_cols = [col for col in yield_df.columns if col != 'Period']
df['Avg_Yield'] = df[yield_country_cols].mean(axis=1)

# Lag and Rolling features
df['Avg_Yield_lag1'] = df['Avg_Yield'].shift(1)
df['Avg_Yield_roll3'] = df['Avg_Yield'].rolling(window=3).mean().shift(1)
df['Avg_Yield_roll6'] = df['Avg_Yield'].rolling(window=6).mean().shift(1)

# Drop only rows with NaNs from the new features
df.dropna(subset=['Avg_Yield_lag1', 'Avg_Yield_roll3', 'Avg_Yield_roll6'], inplace=True)



KeyError: "None of [Index(['Indonesia', 'Malaysia', 'Nigeria', 'Thailand', 'Colombia', 'Ghana',\n       'Honduras', 'Papua New Guinea', 'Other Countries'],\n      dtype='object')] are in the [columns]"

In [ ]:

# Define features and target
X = df.drop(columns=['Yield', 'Date'])
y = df['Yield']

# Scale features
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)


In [ ]:

model = ExtraTreesRegressor(n_estimators=100, random_state=42)
model.fit(X_scaled, y)

# Plot feature importance
feat_importances = pd.Series(model.feature_importances_, index=X.columns)
important_features = feat_importances[feat_importances > feat_importances.mean()].index.tolist()

plt.figure(figsize=(10, 5))
feat_importances.sort_values().plot(kind='barh')
plt.title('Feature Importances')
plt.tight_layout()
plt.show()

# Keep only important features
X_scaled = X_scaled[important_features]


In [ ]:

# Additional visualizations: feature distribution plots
import matplotlib.gridspec as gridspec

def plot_feature_distributions(dataframe):
    features = dataframe.columns
    n_cols = 3
    n_rows = int(np.ceil(len(features) / n_cols))
    fig = plt.figure(figsize=(15, n_rows * 4))
    gs = gridspec.GridSpec(n_rows, n_cols)

    for i, feature in enumerate(features):
        ax = fig.add_subplot(gs[i])
        sns.histplot(dataframe[feature], kde=True, ax=ax)
        ax.set_title(f'Distribution: {feature}')

    plt.tight_layout()
    plt.show()

print("Feature Distributions:")
plot_feature_distributions(X_scaled)


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)
cv = KFold(n_splits=10, shuffle=True, random_state=42)


In [ ]:

# Extra Trees
et_params = {'n_estimators': [50, 100, 150], 'max_depth': [5, 10, None]}
et_grid = GridSearchCV(ExtraTreesRegressor(random_state=42), et_params, cv=cv, scoring='r2')
et_grid.fit(X_train, y_train)

# AdaBoost
ada_params = {'n_estimators': [50, 100], 'learning_rate': [0.01, 0.1, 1.0]}
ada_grid = GridSearchCV(AdaBoostRegressor(random_state=42), ada_params, cv=cv, scoring='r2')
ada_grid.fit(X_train, y_train)


In [ ]:

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    print("MAE:", mean_absolute_error(y_test, y_pred))
    print("MSE:", mean_squared_error(y_test, y_pred))
    print("RMSE:", mean_squared_error(y_test, y_pred, squared=False))
    print("R2 Score:", r2_score(y_test, y_pred))
    print("MAPE:", mean_absolute_percentage_error(y_test, y_pred))
    
    plt.figure(figsize=(6, 4))
    sns.residplot(x=y_test, y=y_pred, lowess=True, line_kws={'color': 'red'})
    plt.xlabel('Actual')
    plt.ylabel('Residuals')
    plt.title('Residual Plot')
    plt.tight_layout()
    plt.show()

print("Extra Trees Evaluation:")
evaluate_model(et_grid.best_estimator_, X_test, y_test)

print("\nAdaBoost Evaluation:")
evaluate_model(ada_grid.best_estimator_, X_test, y_test)


In [ ]:

et_score = r2_score(y_test, et_grid.predict(X_test))
ada_score = r2_score(y_test, ada_grid.predict(X_test))

print(f"Extra Trees R²: {et_score:.4f}")
print(f"AdaBoost R²: {ada_score:.4f}")

print("\nBest model:", "Extra Trees" if et_score > ada_score else "AdaBoost")
